# Task B – MFCD vs VCD (Self-contained)
This notebook clones the repo, installs dependencies, downloads COCO val2014, runs VCD and MFCD, then prints a comparison table.

In [1]:
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [11]:
import os
ROOT = '/content/VCD_project'
REPO_URL = 'https://github.com/maxheef/ECE209_Project1.git'
BRANCH = 'main'
COCO_VAL = '/content/datasets/coco/val2014'
MODEL_PATH = 'liuhaotian/llava-v1.5-7b'
SEED = 55

# MFCD parameters (from MFCD readme)
MFC_HIGH_ALPHA = 1.0
MFC_LOW_ALPHA = 1.0
MFC_BETA = 0.3
MFC_HIGH_PASS_CUTOFF = 0.1
MFC_LOW_PASS_CUTOFF = 0.9
MFC_FILTER_TYPE = 'gaussian'
print('ROOT=', ROOT)

ROOT= /content/VCD_project


In [12]:
%%bash
set -euo pipefail
cd /content
if [ -d /content/VCD_project/.git ]; then
  git -C /content/VCD_project fetch --depth 1 origin ${BRANCH:-main}
  git -C /content/VCD_project reset --hard FETCH_HEAD
else
  rm -rf /content/VCD_project
  git clone --depth 1 --branch ${BRANCH:-main} ${REPO_URL:-https://github.com/maxheef/ECE209_Project1.git} /content/VCD_project
fi

if [ ! -d /content/VCD_project/originalMFCD/mfcd ]; then
  echo 'Missing /content/VCD_project/originalMFCD/mfcd (check repo contents)'.
  find /content/VCD_project -maxdepth 3 -type d | sed -n '1,120p'
  exit 1
fi


HEAD is now at 6620d7b flash removed from requirements


From https://github.com/maxheef/ECE209_Project1
 * branch            main       -> FETCH_HEAD


In [ ]:
import os
import subprocess

# --- Configuration ---
ENV_NAME = "vcd310" # Updated name for clarity
CONDA_PATH = "/usr/local/miniconda"
ENV_PATH = f"{CONDA_PATH}/envs/{ENV_NAME}"
PYTHON_BIN = f"{ENV_PATH}/bin/python"
PIP_BIN = f"{ENV_PATH}/bin/pip"
REQ_FILE = "/content/VCD_project/requirements.txt"

def run_cmd(cmd):
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    if process.returncode != 0:
        raise Exception(f"Command failed: {cmd}")

# 1. Install Miniconda
if not os.path.exists(CONDA_PATH):
    print("Installing Miniconda...")
    run_cmd("wget -q https://repo.anaconda.com -O /tmp/miniconda.sh")
    run_cmd(f"bash /tmp/miniconda.sh -b -p {CONDA_PATH}")

# 2. Create the Environment with Python 3.10
if not os.path.exists(ENV_PATH):
    print(f"Creating Conda environment: {ENV_NAME} with Python 3.10...")
    run_cmd(f"{CONDA_PATH}/bin/conda create -n {ENV_NAME} python=3.10 -y")

# 3. Install Requirements
if os.path.exists(REQ_FILE):
    print(f"Installing Blackwell-compatible requirements...")
    # Using the cu128 index for Torch/Xformers
    run_cmd(f"{PIP_BIN} install --no-cache-dir -r {REQ_FILE}")
    
    # Install FlashAttention separately to handle potential build needs
    print("Installing FlashAttention-2...")
    run_cmd(f"{PIP_BIN} install flash-attn --no-build-isolation")
    
    run_cmd(f"{PIP_BIN} install shortuuid==1.0.13")

# 4. Save path and Verify
with open("/tmp/mytasks_python_bin.txt", "w") as f:
    f.write(PYTHON_BIN)

print("\n--- Verification ---")
run_cmd(f"{PYTHON_BIN} -c 'import torch; print(f\"Compute Capability: {torch.cuda.get_device_capability()}\")'")


Installing requirements from /content/VCD_project/requirements.txt...
Looking in indexes: https://download.pytorch.org/whl/cu128
ERROR: Ignored the following versions that require a different python version: 2.1.0 Requires-Python >=3.10; 2.1.1 Requires-Python >=3.10; 2.1.2 Requires-Python >=3.10; 2.1.3 Requires-Python >=3.10; 2.2.0 Requires-Python >=3.10; 2.2.1 Requires-Python >=3.10; 2.2.2 Requires-Python >=3.10; 2.2.3 Requires-Python >=3.10; 2.2.4 Requires-Python >=3.10; 2.2.5 Requires-Python >=3.10; 2.2.6 Requires-Python >=3.10; 2.3.0 Requires-Python >=3.11; 2.3.1 Requires-Python >=3.11; 2.3.2 Requires-Python >=3.11; 2.3.3 Requires-Python >=3.11; 2.3.4 Requires-Python >=3.11; 2.3.5 Requires-Python >=3.11; 2.4.0rc1 Requires-Python >=3.11
ERROR: Could not find a version that satisfies the requirement pydantic==2.11.0 (from versions: none)
ERROR: No matching distribution found for pydantic==2.11.0


Exception: Command failed: /usr/local/miniconda/envs/vcd39/bin/pip install --no-cache-dir -r /content/VCD_project/requirements.txt

In [6]:
%%bash
set -euo pipefail
mkdir -p /content/datasets/coco
cd /content/datasets/coco
if [ ! -d val2014 ]; then
  wget -q http://images.cocodataset.org/zips/val2014.zip
  unzip -q val2014.zip
fi
ls -la /content/datasets/coco/val2014 | sed -n '1,8p'


total 6564320
drwxrwxr-x 2 root root 2281472 Aug 16  2014 .
drwxr-xr-x 3 root root    4096 Mar 10 22:55 ..
-rw-rw-r-- 1 root root  213308 Aug 16  2014 COCO_val2014_000000000042.jpg
-rw-rw-r-- 1 root root  383651 Aug 16  2014 COCO_val2014_000000000073.jpg
-rw-rw-r-- 1 root root  176151 Aug 16  2014 COCO_val2014_000000000074.jpg
-rw-rw-r-- 1 root root  163607 Aug 16  2014 COCO_val2014_000000000133.jpg
-rw-rw-r-- 1 root root  105082 Aug 16  2014 COCO_val2014_000000000136.jpg


In [7]:
%%bash
set -euo pipefail
PYBIN=$(cat /tmp/mytasks_python_bin.txt)
PYTHON_BIN="$PYBIN" bash /content/VCD_project/myTasks/run_table1.sh \
  liuhaotian/llava-v1.5-7b \
  /content/datasets/coco/val2014 \
  55


cat: /tmp/mytasks_python_bin.txt: No such file or directory


CalledProcessError: Command 'b'set -euo pipefail\nPYBIN=$(cat /tmp/mytasks_python_bin.txt)\nPYTHON_BIN="$PYBIN" bash /content/VCD_project/myTasks/run_table1.sh \\\n  liuhaotian/llava-v1.5-7b \\\n  /content/datasets/coco/val2014 \\\n  55\n'' returned non-zero exit status 1.

In [ ]:
%%bash
set -euo pipefail
PYBIN=$(cat /tmp/mytasks_python_bin.txt)
$PYBIN - <<'PY'
import os
import json
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset

ROOT = '/content/VCD_project'
ORIG = f'{ROOT}/originalProject'
MFCD = f'{ROOT}/originalMFCD/mfcd'
COCO_VAL = '/content/datasets/coco/val2014'
MODEL_PATH = 'liuhaotian/llava-v1.5-7b'

MFC_HIGH_ALPHA = 1.0
MFC_LOW_ALPHA = 1.0
MFC_BETA = 0.3
MFC_HIGH_PASS_CUTOFF = 0.1
MFC_LOW_PASS_CUTOFF = 0.9
MFC_FILTER_TYPE = 'gaussian'

os.environ['PYTHONPATH'] = MFCD + ':' + os.environ.get('PYTHONPATH','')

from eval_utils import prepare_for_generate, generate_responses, batch_prepare_data
from eval_utils import MODEL_INFOS
from eval.pope.eval import eval as pope_eval, recorder

class POPELocal(Dataset):
    def __init__(self, json_path, image_root):
        self.rows = [json.loads(line) for line in open(json_path, 'r')]
        self.image_root = Path(image_root)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        img = Image.open(self.image_root / row['image']).convert('RGB')
        return {
            'question': row['text'],
            'image': img,
            'label': row['label'].lower(),
        }

def run_mfcd(pope_type):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    class Args: pass
    args = Args()
    args.model_name_or_path = MODEL_PATH
    args.model_type = 'llava-1.5'
    args.num_workers = 1
    args.eval_batch_size = 1
    args.device = 'cuda:0'
    args.temperature = 1.2
    args.top_p = 1.0
    args.top_k = 50
    args.max_new_tokens = 2048
    args.mfc_high_alpha = MFC_HIGH_ALPHA
    args.mfc_low_alpha = MFC_LOW_ALPHA
    args.mfc_beta = MFC_BETA
    args.mfc_jsd = False
    args.mfc_entropy = False
    args.mfc_high_pass_cutoff = MFC_HIGH_PASS_CUTOFF
    args.mfc_low_pass_cutoff = MFC_LOW_PASS_CUTOFF
    args.mfc_filter_type = MFC_FILTER_TYPE

    processor, generation_config, _ = prepare_for_generate(args=args, model_info=MODEL_INFOS['llava-1.5'], device=device)

    pope_json = f"{ORIG}/experiments/data/POPE/coco/coco_pope_{pope_type}.json"
    dataset = POPELocal(pope_json, COCO_VAL)

    prompts = batch_prepare_data(args=args, dataset=dataset, processor=processor, question_column_name='question', image_column_name='image')

    model_info = MODEL_INFOS['llava-1.5']
    model = model_info.model_class.from_pretrained(MODEL_PATH, **model_info.model_config).to(device)

    responses = generate_responses(model=model, processor=processor, prompts=prompts, generation_config=generation_config)

    labels = [1 if d['label'] == 'yes' else 0 for d in dataset]
    preds = recorder(responses)
    return pope_eval(preds, labels)

mfcd_random = run_mfcd('random')
mfcd_popular = run_mfcd('popular')
print('MFCD random:', mfcd_random)
print('MFCD popular:', mfcd_popular)
PY


In [ ]:
%%bash
set -euo pipefail
PYBIN=$(cat /tmp/mytasks_python_bin.txt)
$PYBIN - <<'PY'
import re
from pathlib import Path
P1 = '/content/VCD_project/myTasks'
SEED = 55
out_dir = Path(f'{P1}/output')
def parse_metrics(path: Path):
    text = path.read_text()
    def g(name):
        m = re.search(rf"^{name}:\s*([0-9.]+)", text, re.MULTILINE)
        return float(m.group(1)) if m else float('nan')
    return {
        'accuracy': g('Accuracy'),
        'precision': g('Precision'),
        'recall': g('Recall'),
        'f1': g('F1'),
    }
vcd_random = parse_metrics(out_dir / f'metrics_random_vcd_seed{SEED}.txt')
vcd_popular = parse_metrics(out_dir / f'metrics_popular_vcd_seed{SEED}.txt')
print('| Split | Method | Accuracy | Precision | Recall | F1 |')
print('|---|---|---:|---:|---:|---:|')
print(f'| random | VCD | {vcd_random["accuracy"]:.4f} | {vcd_random["precision"]:.4f} | {vcd_random["recall"]:.4f} | {vcd_random["f1"]:.4f} |')
print(f'| popular | VCD | {vcd_popular["accuracy"]:.4f} | {vcd_popular["precision"]:.4f} | {vcd_popular["recall"]:.4f} | {vcd_popular["f1"]:.4f} |')
print('Add MFCD rows from previous cell output.')
PY
